<a href="https://colab.research.google.com/github/smkim0508/cos484-notes/blob/main/A3P3_Neural_Machine_Translation_(COS484_S2026).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook for Programming in Problem 3
Welcome to the programming portion of the assignment! Each assignment throughout the semester will have a written portion and a programming portion. We will be using [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb#recent=true), so if you have never used it before, take a quick look through this introduction: [Working with Google Colab](https://docs.google.com/document/d/1LlnXoOblXwW3YX-0yG_5seTXJsb3kRdMMRYqs8Qqum4/edit?usp=sharing).

## Learning Objectives
In this problem, we will use [PyTorch](https://pytorch.org/) to implement a sequence-to-sequence (seq2seq) transformer model to build a nerual machine translation (NMT) system, which translates from French to English.

## Installing Packages

Install PyTorch using pip. See [https://pytorch.org/](https://pytorch.org/) if you want to install it on your computer.
In addition, we will also be needing [huggingface](https://huggingface.co/)'s `transformers` and `datasets` libraries, and [nltk](https://www.nltk.org/) to compute the BLEU score.

In [2]:
import torch

## Download NMT data

You need to first download and then unzip the data for NMT, which contains pairs of parallel sentences from the link below:
* Resources: https://princeton-nlp.github.io/cos484/assignments/a3/resources.zip

## Data Loading and Tokenization
You will need to write code to load and tokenize the data for NMT.


The parallel data `parallel_en_fr_corpus` is provided as huggingface datasets, one for each split of `train`, `validation` and `test`. You can load it via the `load_from_disk` method and inspect its features. If you'd like to know more about these dataset objects, have a look at [this tutorial](https://huggingface.co/docs/datasets/access).

You are also provided with two pre-trained tokenizers for the source and target languages, `tokenizer_fr` and `tokenizer_en`, respectively, which can be loaded with the hugginface transformers library. [This tutorial](https://huggingface.co/docs/transformers/preprocessing#natural-language-processing) provides an introduction to using pre-trained tokenizers and the powerful `AutoTokenizer` class. The tokenizers are based on byte-pair encodings which break words into smaller units. This is aimed at reducing the sparsity of words, as subwords can be shared between different rare words. If you are interested in learning more, see the paper [Neural Machine Translation of Rare Words with Subword Units](https://www.aclweb.org/anthology/P16-1162.pdf).

You should implement a function to convert the entire dataset to token ids.

## Transformer model for NMT

You should implement a encoder-decoder transformer model. This includes:
* **MultiHeadAttention Layers**: A class that initializes the multi-head attention layer and has a forward function to compute scaled dot-product attention and returns the output of the attention layer.
* **Embedding Layers**: A class that defines both the token embeddings and positional embeddings, which are added together to form the final embedding. This class should have functions to compute the logits for the next token prediction given the decoder output, and to compute the embeddings for the input tokens.
* **Transformer Blocks**: A class that defins a single Transformer block including the multi-head attention layers and feedforward layers, which can be either for the encoder or the decoder. This class should have a forward function that works either for the encoder or the decoder.
* **EncoderDecoderModel**: A class that defines a encoder-decoder transformer model which can be used for NMT.

For your MultiHeadAttention, your `MultiHeadAttention.forward()` should return a tuple of `(layer_output, attention_weights)`, where `attention_weights` has shape `(batch_size, num_heads, num_query_tokens, num_key_tokens)`. Your `forward_decoder()` should accept an optional flag `return_attention_weights: bool = False` and, when set to `True`, return the cross-attention weights from each decoder layer as a list of tensors in addition to the logits. You will need this to extract and visualize the cross-attention weights in Question (d).

For your EncoderDecoderModel, you can use these architecture hyperparameters without modifications:
* hidden_size: 32
* intermediate_size: 128 (32 * 4)
* num_attention_heads: 4
* num_encoder_layers: 3
* num_decoder_layers: 3
* max_sequence_length: 32
* hidden_dropout_prob: 0.1
* FFN activation: ReLU
* norm placement: post-norm
* attention weight dropout: none
* embedding dropout: none
* weight tying: none

Before moving on to the other modules, you should also implement a sanity check for your attention implementation:
1. Check the dimensions of the output of the layer and
2. Plot the attention weights to some toy embedding inputs.


You can assume that the last token in the encoder and the last two tokens in the decoder are pad tokens.

## Train the model

You should implement the logic for your model to train on the parallel tokenized corpus. Before you start training models, you should implement and test the model and its sub-modules, especially the attention.


Use these hyperparameters for your training loop.
* batch_size: 32
* optimizer: Adam (default betas)
* learning_rate: 1e-3
* max_epochs: 15
* log_every: 10 steps
* valid_niter: 100 steps
* loss: cross-entropy, no label smoothing
* loss normalization: per token

Set a random seed, so you obtain the same output model if you run this cell again. With a reasonable implementation, this should take about 16 minutes on CPU / 3 minutes on GPU and you should achieve a validation perplexity of below 10. Note that different implementation and variation across different colab cpus may yield different speeds!

## Evaluate the model

You have now trained a seq2seq model for the NMT task. Now evaluate the model on the test set by generating translations with beam search and comparing them to the gold translations using the corpus-level BLEU score from `nltk.translate.bleu_score.corpus_bleu`.

Use these hyperparameters for your evaluations.
* beam_width: 5
* max_decoding_length: 32

Look at some examples. What do you think of the quality of the translations? Are these grammatical English sentences? Can you identify any common mistakes?

# Questions

**(b) (2 points)**

- **(i)
    What vocabulary size are we using for the source and target language?**

- **(ii)
    Approximately how many source and target tokens are on average contained in a training batch? What proportion of these tokens are `<pad>` tokens on average?**


TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)

Code for sub-part (b)


**(c) (2 points)
Manually look at some results and compare them with the gold answers. What do you think of the quality of the translations? Are these grammatical English sentences? Can you identify any common mistakes?**

TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)



**(d) (10 points) Extract and visualize the *cross-attention* weights from the *last decoder layer*, averaged across all attention heads. This should produce a single 2-D heatmap per example, where the rows correspond to the generated English (target) tokens and the columns correspond to the French (source) tokens.**

Produce attention heatmaps for the following two French sentences:
* Le chat noir est assis sur le tapis rouge.
* Elle n'a pas encore pris une décision importante concernant son avenir.

For each example, answer the following questions:
* Does the attention pattern resemble a word-level alignment between the source and target sentences? Where does it deviate from a diagonal pattern, and why might that be the case? *(Hint: Consider word order differences between French and English.)*
* Does the attention heatmap reveal anything about how the model handles the translation? For instance, are there tokens where attention is diffuse or concentrated, and can you relate this to the structure of the sentence?


TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)

Code for sub-part (d)

# LLM Prompts

If you used an AI tool to complete any part of this assignment, please paste all prompts you used to produce your final code/responses in the box below and answer the following reflection question.

Prompts Used:
*   
*   



**Reflection: What parts of the AI generated output required modification or improvement? Describe the feedback you gave the tool to produce your final output or any changes you had to make on your own.**

TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)